In [ ]:
from datasets import load_dataset, load_from_disk, DatasetDict, Dataset


dataset_name = "HuggingFaceH4/ultrafeedback_binarized"

raw_dataset = load_dataset(dataset_name)['train_prefs']
test_dataset = load_dataset(dataset_name)['test_prefs']


# raw_dataset = Dataset.from_dict({
#     'prompt': raw_dataset['prompt'] *2,
#     'prompt_id': list(range(len(raw_dataset['prompt'] *2))),
#     'messages': raw_dataset['chosen'] + raw_dataset['rejected']
#     })

raw_dataset = Dataset.from_dict({
    'prompt': raw_dataset['prompt'],
    'prompt_id': list(range(len(raw_dataset['prompt']))),
    # 'messages': raw_dataset['chosen'],
    'messages': raw_dataset['rejected'],
    })

required_columns = ['prompt', 'prompt_id', 'messages']

raw_dataset = raw_dataset.remove_columns([column_name for column_name in raw_dataset.column_names if column_name not in required_columns ])
test_dataset = test_dataset.remove_columns([column_name for column_name in test_dataset.column_names if column_name not in required_columns ])


# new_dataset = DatasetDict({
#     'train': raw_dataset.shuffle(seed=42),
#     'test': test_dataset
# })



raw_dataset.to_json("ultrafeedback-sft-chosen.json")
# new_dataset.push_to_hub("jlpang888/ultrafeedback-sft-identical-pairs-7387-rejected")
# new_dataset.push_to_hub("jlpang888/ultrafeedback-sft-chosen")

In [ ]:
from datasets import load_dataset, Dataset

dataset_name = "jlpang888/ultrafeedback_sorted_score_diff"
raw_dataset = load_dataset(dataset_name)['train']

# 筛选 score_chosen - score_rejected <= 0.5
filtered = raw_dataset.filter(lambda x: x['score_chosen'] - x['score_rejected'] < 0.5)

print(f"原始样本数: {len(raw_dataset)}")
print(f"筛选后样本数: {len(filtered)}")
print(filtered)

raw_dataset = Dataset.from_dict({
    'prompt': raw_dataset['prompt'],
    'prompt_id': list(range(len(raw_dataset['prompt']))),
    # 'messages': raw_dataset['chosen'],
    'messages': raw_dataset['rejected'],
    })

required_columns = ['prompt', 'prompt_id', 'messages']


Filter: 100%|██████████| 61135/61135 [00:01<00:00, 32944.62 examples/s]

原始样本数: 61135
筛选后样本数: 7387
Dataset({
    features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
    num_rows: 7387
})


In [ ]:
# 保存筛选后的 DPO 格式数据集（score_diff < 0.5 的 7387 个样本）
from datasets import DatasetDict

dpo_filtered = DatasetDict({
    'train': filtered,
    'test': filtered.select(range(100)),  # 随便取100个防报错
})

dpo_filtered.save_to_disk("/home/jlpang/QualityDPO/datasets/ultrafeedback_sorted_score_diff_difficult_7387")
print(f"Train: {len(filtered)}, Test: 100")


In [ ]:
# 转成 SFT 格式: {'prompt', 'prompt_id', 'messages'}

# 1. chosen response only
sft_chosen = Dataset.from_dict({
    'prompt': filtered['prompt'],
    'prompt_id': list(range(len(filtered))),
    'messages': filtered['chosen'],
})

# 2. rejected response only
sft_rejected = Dataset.from_dict({
    'prompt': filtered['prompt'],
    'prompt_id': list(range(len(filtered))),
    'messages': filtered['rejected'],
})

# 3. chosen + rejected (每个 prompt 出现两次)
sft_both = Dataset.from_dict({
    'prompt': filtered['prompt'] + filtered['prompt'],
    'prompt_id': list(range(len(filtered) * 2)),
    'messages': filtered['chosen'] + filtered['rejected'],
})

print(f"SFT chosen: {len(sft_chosen)}")
print(f"SFT rejected: {len(sft_rejected)}")
print(f"SFT both: {len(sft_both)}")

# 保存为 JSON
sft_chosen.to_json("ultrafeedback_sft_difficult_pair_chosen.json")
sft_rejected.to_json("ultrafeedback_sft_difficult_pair_rejected.json")
sft_both.to_json("ultrafeedback_sft_difficult_pair_rejected_and_chosen.json")

print("Done!")
